# Tag & Probe: the HLT_Ele27_WPTight_Gsf efficiency with coffea

HLT_Ele27_WPTight_Gsf fires on an isolated electron with $p_T > 27$ GeV passing a tight online ID.
We measure its efficiency in 2016 data with Z → ee events, using the tag & probe method.

**Tag** electron:
- $p_T > 30$ GeV and $|\eta| < 2.5$
- not in the ECAL barrel-endcap transition region $1.4442 < |\eta| < 1.566$
- passes the tight cut-based ID
- matched ($\Delta R < 0.1$) to an HLT electron that fired HLT_Ele27_WPTight_Gsf

**Probe** electron:
- opposite charge to the tag
- passes the same tight cut-based ID
- the tag-probe invariant mass is less than 30 GeV away from the Z mass (91.1876 GeV)
- $|\eta| < 2.5$ and not in the transition region, like the tag

A probe is **passing** if it is also matched ($\Delta R < 0.1$) to an HLT electron that fired HLT_Ele27_WPTight_Gsf,
and **failing** otherwise. In each probe $p_T$ bin:

$$\text{efficiency} = \frac{\text{passing}}{\text{passing} + \text{failing}}$$

Fill in every function that raises `NotImplementedError`, from top to bottom. The last cell runs the whole measurement.

## Setup
Imports, the input file and the probe $p_T$ bin edges. To run on another file, change `FILENAME`.

In [ ]:
import awkward as ak
import hist
import matplotlib.pyplot as plt
import numpy as np
from coffea.nanoevents import NanoEventsFactory
from hist import intervals

FILENAME = "/data/mandi26/cms-opendata-singleelectron-small.root"

# Probe pT bin edges in GeV
PT_BINS = [5, 10, 12, 14, 16, 18, 20, 23, 26, 28, 30, 32, 34, 36, 38, 40, 45, 50, 60, 80, 100, 150, 250, 400]

## Step 1: Preselect the events
The tag has to fire the trigger, so we keep only the events that fired HLT_Ele27_WPTight_Gsf. A tag-probe pair also needs at least two electrons.

In [ ]:
def select_events(events):
    """Keep only the events that fired HLT_Ele27_WPTight_Gsf and have at least two electrons."""
    raise NotImplementedError

## Step 2: Match electrons to the trigger
`TrigObj` holds the objects that the HLT built online. An offline electron is trigger matched if an HLT electron that passed HLT_Ele27_WPTight_Gsf is close to it. The last cell runs it on all the electrons and stores the result in two electron fields: `fires_tag_trigger`, which the tag must have, and `fires_probe_trigger`, which makes a probe pass. Here both are HLT_Ele27_WPTight_Gsf; to measure another trigger, only `fires_probe_trigger` changes.

In [ ]:
def trigger_match(electrons, trigobjs):
    """Return True for each electron matched to an HLT_Ele27_WPTight_Gsf trigger object.

    Use only trigger objects that are electrons (|id| == 11), have pt > 27 GeV and
    have bit 1 of filterBits set ("1e (WPTight)").
    An electron is matched if at least one of them is within dR < 0.1.
    The result has one True/False per electron, the same shape as `electrons`.

    Hint: electrons.metric_table(objects) gives the dR between every electron and every object.
    """
    raise NotImplementedError

## Step 3: Make tag-probe pairs
Each Z → ee candidate is a pair of electrons: one is the tag, the other the probe. How you pair them is your choice, the rest of the code works with any of them. Can both electrons of an event be a probe?

In [ ]:
def make_pairs(electrons):
    """Build tag-probe pairs out of the electrons.

    Return an awkward array of records with the fields "tag" and "probe".
    Tag and probe must be different electrons. How you pair them is up to you,
    the rest of the code works for any choice.
    """
    raise NotImplementedError

## Step 4: Select the tags
The tag has to be a very clean electron, so that the pair really comes from a Z. Your function gets the tags (`pairs.tag`) and returns True/False for each of them.

In [ ]:
def select_tags(tags):
    """Return True for the electrons that pass the tag selection.

    pT > 30 GeV, |eta| < 2.5, not in 1.4442 < |eta| < 1.566, tight cut-based ID and firing
    the tag trigger (the "fires_tag_trigger" field, added to the electrons before making the pairs).
    """
    raise NotImplementedError

## Step 5: Select the probes
The probe passes the same offline ID as the tag, but has no trigger requirement: the trigger is what we measure.

In [ ]:
def select_probes(probes):
    """Return True for the electrons that pass the probe selection.

    Tight cut-based ID, |eta| < 2.5 and not in 1.4442 < |eta| < 1.566.
    No pT cut: we measure the efficiency as a function of the probe pT.
    """
    raise NotImplementedError

## Step 6: Keep the Z candidates
A pair with opposite charges and a mass close to the Z mass is very likely a real Z → ee decay, so its probe is very likely a real electron.

In [ ]:
def select_z_candidates(pairs):
    """Return True for the tag-probe pairs that look like a Z -> ee decay.

    Tag and probe have opposite charge and their invariant mass is less than 30 GeV
    away from the Z mass (91.1876 GeV).
    """
    raise NotImplementedError

## Step 7: Fill the histograms
Split the probes into passing and failing and histogram their $p_T$.

In [ ]:
def fill_histograms(pairs):
    """Fill one histogram of the probe pT with the passing probes and one with the failing probes.

    A probe passes if it fires the trigger we measure (its "fires_probe_trigger" field).
    Bin the histograms in PT_BINS. This must work whatever shape make_pairs returns.
    Return the two hist.Hist histograms (h_pass, h_fail).
    """
    raise NotImplementedError

## Step 8: Compute the efficiency
This one is given, just run the cell. The uncertainty is the Clopper-Pearson interval, the usual choice for an efficiency.

In [ ]:
def compute_efficiency(h_pass, h_fail):
    """Return the efficiency passing / (passing + failing) in each bin and its uncertainty.

    This function is given, there is nothing to fill in. The uncertainty is the Clopper-Pearson
    interval, returned as the (lower, upper) errors.
    """
    passing = h_pass.values()
    total = h_pass.values() + h_fail.values()
    # An empty bin gives 0 / 0 = nan, which is simply not drawn
    with np.errstate(divide="ignore", invalid="ignore"):
        efficiency = passing / total
        errors = intervals.ratio_uncertainty(passing, total, uncertainty_type="efficiency")
    return efficiency, errors

## Step 9: Plot the efficiency
This one is given too, just run the cell.

In [ ]:
def plot_efficiency(bin_edges, efficiency, errors):
    """Plot the efficiency versus the probe pT, show it and save it as efficiency.png.

    This function is given, there is nothing to fill in. Each bin is drawn as one point at the
    bin center, with x error = half the bin width and y error = `errors`.
    """
    bin_edges = np.array(bin_edges)
    centers = (bin_edges[1:] + bin_edges[:-1]) / 2
    half_widths = (bin_edges[1:] - bin_edges[:-1]) / 2

    fig, ax = plt.subplots()
    ax.errorbar(centers, efficiency, xerr=half_widths, yerr=errors, fmt="o", label="HLT_Ele27_WPTight_Gsf")
    ax.set_xscale("log")
    ax.set_xlim(5, 400)
    ax.set_ylim(0, 1.2)
    ax.set_xticks([5, 10, 20, 30, 50, 100, 200, 400], ["5", "10", "20", "30", "50", "100", "200", "400"])
    ax.set_xlabel("Offline electron $p_T$ [GeV]")
    ax.set_ylabel("L1T + HLT efficiency")
    ax.legend(loc="lower right")
    fig.savefig("efficiency.png")
    plt.show()

## Run the measurement
This cell loads the events and puts all the steps together. Run it once all the functions above are filled in.

In [ ]:
# Open the file in virtual mode: a branch is only read from the file the first time it is used
events = NanoEventsFactory.from_root({FILENAME: "Events"}, mode="virtual").events()
print("Events in the file:", len(events))

events = select_events(events)
print("Events after the preselection:", len(events))

# Trigger match every electron before making the pairs and store the results as electron fields,
# so tags and probes carry them along however make_pairs builds the pairs. The tag must fire the
# tag trigger, and a probe passes if it fires the trigger we measure. Here both are
# HLT_Ele27_WPTight_Gsf; to measure another trigger, only fires_probe_trigger changes.
events["Electron", "fires_tag_trigger"] = trigger_match(events.Electron, events.TrigObj)
events["Electron", "fires_probe_trigger"] = trigger_match(events.Electron, events.TrigObj)

pairs = make_pairs(events.Electron)
pairs = pairs[select_tags(pairs.tag)]
pairs = pairs[select_probes(pairs.probe)]
pairs = pairs[select_z_candidates(pairs)]
print("Tag-probe pairs:", ak.count(pairs.probe.pt, axis=None))

h_pass, h_fail = fill_histograms(pairs)
print("Passing probes:", int(h_pass.sum(flow=True)))
print("Failing probes:", int(h_fail.sum(flow=True)))

efficiency, errors = compute_efficiency(h_pass, h_fail)
plot_efficiency(h_pass.axes[0].edges, efficiency, errors)
print("Saved the plot to efficiency.png")

Different pairing choices in Step 3 give different numbers of probes. Do the efficiencies agree?

## Scaling up
So far we ran on a single file. Now run your measurement on the files below with a coffea processor and executor, as in the coffea lesson:

- Put the steps of the last cell in the `process` method of a processor. It should return the passing and failing histograms of the dataset it runs on.
- Define a fileset with the two datasets below. Give each dataset an `is_mc` entry in its metadata (False for data, True for DY).
- Run it with a `processor.Runner` and a `FuturesExecutor` with 4 workers.

| Dataset | Files |
|---|---|
| Data | `/data/mandi26/cms-opendata-singleelectron-small.root`<br>`/data/mandi26/cms-opendata-singleelectron-1.root`<br>`/data/mandi26/cms-opendata-singleelectron-2.root` |
| DY (Z → ee simulation) | `/data/mandi26/cms-opendata-dytollamcatnlofxfx-1.root`<br>`/data/mandi26/cms-opendata-dytollamcatnlofxfx-2.root` |

In [ ]:
# Your code here

## Data and simulation efficiencies
Compute the efficiency of both datasets with `compute_efficiency` and plot the two curves in one figure. The code in `plot_efficiency` is a good starting point.

In [ ]:
# Your code here

## Scale factors
The scale factor in each $p_T$ bin is $SF = \varepsilon_{data} / \varepsilon_{MC}$. Compute it and plot it with its uncertainty.

The SF uncertainty comes from the uncertainties of both efficiencies, so you need to propagate them. For a ratio they add in quadrature:

$$\sigma_{SF}^2 = \left(\frac{\sigma_{data}}{\varepsilon_{MC}}\right)^2 + \left(\frac{\varepsilon_{data}\,\sigma_{MC}}{\varepsilon_{MC}^2}\right)^2$$

The efficiency uncertainties from `compute_efficiency` are asymmetric (lower, upper): think about which of them go into the upper and which into the lower SF uncertainty. In bins where both efficiencies are 0 you get 0 / 0: set the SF to 1 there.

In [ ]:
# Your code here

## Bonus: closure test
Apply your scale factors to the probes of an independent DY file, `/data/mandi26/cms-opendata-dytollamcatnlofxfx-3.root`. Open it like in the cell that runs the measurement (give it the metadata your processor needs) and call the `process` method of your processor on it directly.

In each $p_T$ bin, give every passing probe the weight $SF$ and every failing probe the weight $(1 - \varepsilon_{data}) / (1 - \varepsilon_{MC})$, and compute the corrected efficiency of this file. Plot the data efficiency, the efficiency of this file and its corrected efficiency in one figure. Does the corrected simulation agree with data? Why is that expected?

In [ ]:
# Your code here